In [ ]:
#This file will be used to write the inference metrics obtained from footnet_inference.py
import sys
sys.path.append("/Users/abhinavarora/Desktop/CadenceCV/ml/src/inference")
from footnet_inference import frame_to_pred_dict, strikefoot_frames, frame_count, duration
import numpy as np
import scipy.signal

In [ ]:
#Clauclating the average time in air per gait cycle and average time in ground contact
def calculate_metrics():
    #Grouping pairs of 1s and 0s from the frame_to_pred_dict to obtain a gait cycle
    frame_preds = []
    for frame in frame_to_pred_dict:
        frame_preds.append(frame_to_pred_dict[frame])

    #(i+1)th - ith entry
    diff = np.diff(frame_preds)
    #contact:1, non-contact: 0. 
    #So, non_contact AFTER contact = 0 - 1 = -1 (takeoff frame)
    takeoff_frames = np.where(diff == -1)[0]
    #So, contact AFTER non-contact = 1 - 0 = 1 (Strikefoot frame)
    landing_frames = np.where(diff == 1)[0]

    flight_times = []
    gct_times = []
    for takeoff in takeoff_frames:
        #Find the next landing after this takeoff. Finding all landings initially to check if there even is a landing 
        #after the take off frames.
        next_landings = landing_frames[landing_frames > takeoff]
        if len(next_landings) > 0:
            next_landing = next_landings[0]
            flight_times.append(next_landing - takeoff)
    
    for landing in landing_frames:
        next_takeoffs = takeoff_frames[takeoff_frames>landing]
        if len(next_takeoffs) > 0:
            next_takeoff = next_takeoff[0]
            gct_times.append(next_takeoff - landing)

    #Divide by frame count to obtain metrics in seconds
    average_flight_time = (np.mean(flight_times) if flight_times else 0)/frame_count
    average_gct = (np.mean(gct_times) if gct_times else 0)/frame_count

    #Cadence is the steps per second
    cadence = len(strikefoot_frames)/duration
